# **6. Предиктивное моделирование (Machine Learning): Классификация сердечно-сосудистых заболеваний**

* __Цель предиктивного моделирования:__ Разработать и обучить модели машинного обучения для автоматизированной диагностики наличия сердечно-сосудистых патологий на основе комплекса клинических предикторов.
* __Задачи предиктивного моделирования:__ Подготовить данные для обучения (Train/Test Split, балансировка классов). Обучить классические ML-алгоритмы (Logistic Regression, Random Forest, Gradient Boosting). Оптимизировать гиперпараметры моделей для максимизации их предсказательной способности с фокусом на метрику Recall (минимизация ложноотрицательных диагнозов).
* __Алгоритм использования:__
  1. __Подготовка данных:__ Бинаризация целевой переменной (`num`), разделение выборки на обучающую и тестовую с сохранением стратификации.
  2. __Обучение базовых моделей (Baseline):__ Обучение моделей "из коробки" для получения отправной точки.
  3. __Оптимизация гиперпараметров:__ Применение GridSearchCV или Optuna для тонкой настройки моделей с использованием кросс-валидации.
  4. __Оценка качества:__ Сравнительный анализ моделей по метрикам ROC-AUC, F1-Score, Precision и Recall. Выбор лучшей модели (Champion Model).

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

# Сторонние библиотеки
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

# Локальные модули
from scripts.logger import setup_logger

# Инициализация логгера
logger = setup_logger("modeling")

# Настройка визуализации
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Константы
RANDOM_STATE = 42  # Установка параметра для воспроизводимости результатов

In [ ]:
# --- ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ ---

try:
    # Загрузка обработанных данных
    data_path = Path("..") / "data" / "processed" / "heart_disease_uci_processed.csv"
    data = pd.read_csv(data_path)

    display(Markdown("#### **Содержание набора данных для ML**"))
    display(data.head())
    display(
        Markdown(
            f"Размерность данных: **{data.shape[0]} строк и {data.shape[1]} столбцов.**"
        )
    )

    # Бинаризация целевой переменной: 0 - здоров, 1 - болен (стадии 1-4)
    data["target"] = (data["num"] > 0).astype(int)

    # Проверка баланса классов
    class_balance = data["target"].value_counts(normalize=True) * 100
    display(Markdown("#### **Баланс классов целевой переменной (target)**"))
    display(data["target"].value_counts().to_frame(name="Количество"))
    display(
        Markdown(
            f"Здоровые (0): **{class_balance[0]:.1f}%** | Больные (1): **{class_balance[1]:.1f}%**"
        )
    )

except FileNotFoundError as e:
    logger.error("File ../data/processed/heart_disease_uci_processed.csv not found.")
    raise e

## **6.1. Подготовка данных: Разделение на обучающую и тестовую выборки (Train/Test Split)**

* __Цель:__ Изолировать часть набора данных для объективной валидации алгоритмов машинного обучения, чтобы предотвратить эффект переобучения (overfitting) и оценить реальную обобщающую способность моделей на новых, ранее не виданных ими данных.
* __Задачи:__ Отделить матрицу независимых предикторов (`X`) от вектора целевой переменной (`y`). Разделить данные на обучающую (Train) и тестовую (Test) выборки в классической пропорции 80% / 20%. Гарантировать статистически идентичное распределение классов в обеих выборках для предотвращения смещения алгоритма (Bias).
* __Алгоритм использования:__
  1. __Формирование матриц:__ Удаление целевой переменной (`target` и `num`) и системных идентификаторов (`id`) из массива признаков.
  2. __Стратифицированное разбиение:__ Использование алгоритма `train_test_split` с параметром `stratify=y`. Этот параметр принудительно сохраняет исходный баланс классов (около 45% здоровых на 55% больных) как в обучающем, так и в тестовом наборах.
  3. __Контрольная валидация:__ Количественная проверка размерностей полученных матриц и процентного соотношения целевого класса для подтверждения математической корректности сплита.

In [ ]:
# --- 6.1.1. РАЗДЕЛЕНИЕ ДАННЫХ (TRAIN / TEST SPLIT) ---

display(Markdown("### **Разделение данных на обучающую и тестовую выборки**"))

# Определение целевой переменной (y) и матрицы признаков (X)
# Удаление исходного 'num' (во избежание утечки данных), 'id' (неинформативный) и нового 'target' (целевой) из X
X = data.drop(columns=["num", "id", "target"])
y = data["target"]

# Разделение данных: 80% на обучение, 20% на тест (с сохранением баланса классов)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

display(
    Markdown(
        f"* **Размер исходных данных:** {X.shape[0]} строк\n"
        f"* **Размер обучающей выборки (Train):** {X_train.shape[0]} строк ({X_train.shape[0] / X.shape[0] * 100:.0f}%)\n"
        f"* **Размер тестовой выборки (Test):** {X_test.shape[0]} строк ({X_test.shape[0] / X.shape[0] * 100:.0f}%)\n"
        f"* **Количество признаков (фичей) для обучения:** {X_train.shape[1]}"
    )
)

# Проверка стратификации (сохранения баланса классов)
train_balance = y_train.value_counts(normalize=True) * 100
test_balance = y_test.value_counts(normalize=True) * 100

display(Markdown("#### **Контроль баланса классов после разделения**"))
balance_df = pd.DataFrame({"Train (%)": train_balance, "Test (%)": test_balance}).round(
    1
)
display(balance_df)

## **6.2. Обучение базовых моделей (Baseline Modeling)**

* __Цель:__ Получить первичную оценку предсказательной способности различных алгоритмов машинного обучения "из коробки" (с гиперпараметрами по умолчанию) для формирования точки отсчета (benchmark) перед этапом оптимизации.
* __Задачи:__ Инициализировать и обучить три архитектурно разных алгоритма: линейную модель (Logistic Regression), бэггинг-ансамбль (Random Forest) и градиентный бустинг (XGBoost). Рассчитать ключевые метрики классификации для каждой модели и провести их сравнительный анализ.
* __Алгоритм использования:__
  1. __Инициализация:__ Создание экземпляров моделей с фиксацией `random_state` для обеспечения полной воспроизводимости результатов.
  2. __Обучение (Fit):__ Передача обучающей матрицы признаков (`X_train`) и вектора ответов (`y_train`) в каждую модель для поиска закономерностей.
  3. __Предикция и Оценка (Predict & Evaluate):__ Генерация предсказаний на отложенной тестовой выборке (`X_test`). Расчет метрик: Accuracy, Precision, Recall, F1-Score и ROC-AUC. 
      * *Примечание: В контексте медицинской диагностики (поиск больных пациентов) метрика **Recall (Чувствительность)** имеет наивысший приоритет, так как цена ложноотрицательного диагноза (пропуск болезни) критически высока.*

In [ ]:
# --- 6.2.1. ОБУЧЕНИЕ И ОЦЕНКА БАЗОВЫХ МОДЕЛЕЙ ---

display(Markdown("### **Обучение и сравнение базовых моделей (Baseline)**"))

# Инициализация словаря для хранения результатов
baseline_results = []

# Инициализация моделей с базовыми настройками
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
}

logger.info("Running training for baseline models...")

# Цикл обучения и оценки каждой модели
for model_name, model in models.items():
    try:
        # 1. Обучение модели на тренировочной выборке
        model.fit(X_train, y_train)

        # 2. Получение предсказаний на тестовой выборке
        y_pred = model.predict(X_test)

        # Получение вероятностей предсказаний (для ROC-AUC)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        # 3. Расчет метрик
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_proba)

        # Сохранение результатов
        baseline_results.append(
            {
                "Модель": model_name,
                "Accuracy": acc,
                "Precision": prec,
                "Recall (Sensitivity)": rec,
                "F1-Score": f1,
                "ROC-AUC": roc_auc,
            }
        )

        logger.info(f"Model {model_name} trained and evaluated successfully.")

    except Exception as e:
        logger.error(f"Error during model training {model_name}: {e}")

# Формирование итоговой таблицы сравнения
results_df = pd.DataFrame(baseline_results)


# Функция для цветового выделения максимальных значений в каждом столбце
def highlight_max(s: pd.Series) -> list:
    is_max = s == s.max()
    return ["color: green; font-weight: bold" if v else "" for v in is_max]


display(Markdown("#### **Сводная таблица метрик классификации (Test Set)**"))
display(Markdown("*Зеленым цветом выделены лучшие показатели по каждой метрике.*"))

# Применение стилей и вывод таблицы
display(
    results_df.style.hide(axis="index")
    .apply(
        highlight_max,
        subset=["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC"],
    )
    .format(
        {
            "Accuracy": "{:.3f}",
            "Precision": "{:.3f}",
            "Recall (Sensitivity)": "{:.3f}",
            "F1-Score": "{:.3f}",
            "ROC-AUC": "{:.3f}",
        }
    )
)

## **6.3. Оптимизация гиперпараметров (Hyperparameter Tuning)**

* __Цель:__ Максимизировать предсказательную способность моделей-лидеров с фокусом на повышение чувствительности (Recall) алгоритма.
* __Задачи:__ Задать сетку гиперпараметров для каждой модели. Использовать метод полного перебора с кросс-валидацией (GridSearchCV) для поиска оптимальной комбинации настроек, предотвращая переобучение.
* __Алгоритм использования:__
  1. __Инициализация GridSearchCV:__ Настройка объекта поиска с указанием модели, сетки параметров (param_grid) и целевой метрики (`scoring='recall'`).
  2. __Кросс-валидация (CV=5):__ Обучение моделей на 5 различных фолдах (разбиениях) обучающей выборки для получения усредненной, объективной оценки качества.

In [ ]:
# --- 6.3.1. НАСТРОЙКА ГИПЕРПАРАМЕТРОВ (GRIDSEARCH CV) ---

display(Markdown("### **Поиск оптимальных гиперпараметров (GridSearchCV)**"))

# 1. Настройка сетки параметров для Logistic Regression
lr_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],  # Сила регуляризации
    "solver": ["liblinear", "lbfgs"],  # Алгоритмы оптимизации
    "max_iter": [1000],
}

# 2. Настройка сетки параметров для Random Forest
rf_param_grid = {
    "n_estimators": [100, 200],  # Количество деревьев
    "max_depth": [None, 5, 10, 15],  # Максимальная глубина деревьев
    "min_samples_split": [2, 5, 10],  # Минимальное число примеров для разделения узла
    "class_weight": ["balanced", None],  # Штраф за ошибки в миноритарном классе
}

logger.info("Running GridSearchCV for Logistic Regression...")
lr_grid = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    param_grid=lr_param_grid,
    scoring="recall",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
lr_grid.fit(X_train, y_train)

logger.info("Running GridSearchCV for Random Forest...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=rf_param_grid,
    scoring="recall",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

# Извлечение лучших моделей
best_lr = lr_grid.best_estimator_
best_rf = rf_grid.best_estimator_

display(Markdown("#### **Лучшие найденные параметры**"))
display(Markdown(f"* **Logistic Regression:** `{lr_grid.best_params_}`"))
display(Markdown(f"* **Random Forest:** `{rf_grid.best_params_}`"))

# --- 6.3.2. ОЦЕНКА ОПТИМИЗИРОВАННЫХ МОДЕЛЕЙ ---

tuned_results = []
best_models = {"Tuned Logistic Regression": best_lr, "Tuned Random Forest": best_rf}

for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    tuned_results.append(
        {
            "Модель": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall (Sensitivity)": recall_score(y_test, y_pred),
            "F1-Score": f1_score(y_test, y_pred),
            "ROC-AUC": roc_auc_score(y_test, y_pred_proba),
        }
    )

# Сравнение с лучшим бейзлайном
tuned_df = pd.DataFrame(tuned_results)
combined_df = pd.concat(
    [results_df.loc[results_df["Модель"] == "Logistic Regression"], tuned_df],
    ignore_index=True,
)

display(Markdown("#### **Сравнение: Baseline против Tuned моделей (Test Set)**"))
display(
    combined_df.style.hide(axis="index")
    .apply(
        highlight_max,
        subset=["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC"],
    )
    .format(
        {
            "Accuracy": "{:.3f}",
            "Precision": "{:.3f}",
            "Recall (Sensitivity)": "{:.3f}",
            "F1-Score": "{:.3f}",
            "ROC-AUC": "{:.3f}",
        }
    )
)

## **6.4. Глубокая оценка лучшей модели (Champion Model Evaluation)**

* __Цель:__ Визуально интерпретировать предсказательную способность модели-победителя (Tuned Random Forest) для детального понимания характера допускаемых ею ошибок.
* __Задачи:__ Построить матрицу ошибок (Confusion Matrix) для оценки соотношения ложноположительных (False Positives) и ложноотрицательных (False Negatives) диагнозов. Визуализировать ROC-кривую (Receiver Operating Characteristic) для оценки баланса между чувствительностью и специфичностью при различных порогах классификации.
* __Алгоритм использования:__
  1. __Матрица ошибок (Confusion Matrix):__ Построение тепловой карты, где по диагонали расположены верные предсказания (True Positives и True Negatives), а вне диагонали — ошибки. Особый фокус на правый верхний квадрат (пропуск болезни).
  2. __ROC-кривая:__ Построение графика зависимости True Positive Rate (Recall) от False Positive Rate. Вычисление площади под кривой (AUC). Идеальная модель имеет кривую, стремящуюся в левый верхний угол (AUC = 1.0).

In [ ]:
# --- 6.4.1. ВИЗУАЛИЗАЦИЯ МАТРИЦЫ ОШИБОК И ROC-КРИВОЙ ---

display(Markdown("### **Визуализация метрик качества: Tuned Random Forest**"))

# Получение предсказаний и вероятностей для лучшей модели (Random Forest)
y_pred_champ = best_rf.predict(X_test)
y_pred_proba_champ = best_rf.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Матрица ошибок (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred_champ)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    annot_kws={"size": 16, "weight": "bold"},
    ax=axes[0],
)

# Настройка осей матрицы ошибок
axes[0].set_title("Матрица ошибок (Confusion Matrix)", fontsize=14, pad=15)
axes[0].set_xlabel("Предсказанный диагноз (Модель)", fontsize=12, labelpad=10)
axes[0].set_ylabel("Фактический диагноз (Истина)", fontsize=12, labelpad=10)
axes[0].set_xticklabels(["Здоров (0)", "Болен (1)"], fontsize=11)
axes[0].set_yticklabels(["Здоров (0)", "Болен (1)"], fontsize=11, va="center")

# Добавление текстовых пояснений к ячейкам
axes[0].text(
    0.5, 0.2, "True Negatives", ha="center", va="center", color="black", fontsize=9
)
axes[0].text(
    1.5, 0.2, "False Positives", ha="center", va="center", color="black", fontsize=9
)
axes[0].text(
    0.5, 1.2, "False Negatives", ha="center", va="center", color="black", fontsize=9
)
axes[0].text(
    1.5, 1.2, "True Positives", ha="center", va="center", color="white", fontsize=9
)

# 2. ROC-кривая
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_champ)
roc_auc = roc_auc_score(y_test, y_pred_proba_champ)

axes[1].plot(
    fpr, tpr, color="crimson", lw=2.5, label=f"ROC-кривая (AUC = {roc_auc:.3f})"
)
axes[1].plot(
    [0, 1],
    [0, 1],
    color="navy",
    lw=2,
    linestyle="--",
    label="Random predictions (AUC = 0.500)",
)

# Настройка осей ROC-кривой
axes[1].set_title("ROC-кривая (Receiver Operating Characteristic)", fontsize=14, pad=15)
axes[1].set_xlabel("False Positive Rate", fontsize=12)
axes[1].set_ylabel("True Positive Rate", fontsize=12)
axes[1].legend(loc="lower right", fontsize=12)
axes[1].grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

# Отчет о классификации
display(
    Markdown(
        "#### **Подробный текстовый отчет о классификации (Classification Report)**"
    )
)

report = classification_report(
    y_test, y_pred_champ, target_names=["Здоров (0)", "Болен (1)"]
)

display(Markdown(f"```text\n{report}\n```"))

# Клинико-статистическая интерпретация
tn, fp, fn, tp = cm.ravel()
display(Markdown("#### **Клинико-статистическая интерпретация результатов**"))
display(
    Markdown(
        f"* **Верно выявленные больные (True Positives):** {tp} пациентов. Модель успешно распознала патологию.\n"
        f"* **Верно определенные здоровые (True Negatives):** {tn} пациентов.\n"
        f"* **Ложная тревога (False Positives):** {fp} здоровых пациентов получили диагноз 'Болен'. С медицинской точки зрения это *допустимая* ошибка.\n"
        f"* **КРИТИЧЕСКАЯ ОШИБКА (False Negatives):** {fn} больных пациентов алгоритм назвал 'Здоровыми'. Наш Recall = {tp / (tp + fn):.3f} (87.3%) означает, что мы пропускаем примерно 1 из 8 больных пациентов."
    )
)

## **6.5. Глубокая оценка и интерпретация моделей (Explainable AI - XAI)**

* __Цель:__ "Распаковать" черный ящик модели-победителя (Random Forest) и понять, какие именно физиологические показатели оказывают наибольшее влияние на алгоритм при постановке кардиологического диагноза.
* __Задачи:__ Извлечь значения глобальной важности признаков (Feature Importance) из обученного ансамбля деревьев. Визуализировать и проранжировать предикторы, сопоставив результаты машинного обучения с нашими предыдущими выводами из корреляционного и дисперсионного анализа.
* __Алгоритм использования:__
  1. __Извлечение метрик:__ Получение атрибута `feature_importances_` из модели `best_rf`.
  2. __Агрегация признаков:__ Поскольку некоторые категориальные переменные были закодированы (One-Hot Encoding) и разбились на несколько колонок, мы должны сгруппировать их обратно (например, сложить важности всех колонок `dataset_...` в единый признак `dataset`).
  3. __Визуализация (Bar Plot):__ Построение горизонтальной столбчатой диаграммы для наглядного ранжирования симптомов от самого критичного к наименее значимому.

In [ ]:
# --- 6.5.1. ГЛОБАЛЬНАЯ ВАЖНОСТЬ ПРИЗНАКОВ (FEATURE IMPORTANCE) ---

display(
    Markdown(
        "### **Визуализация важности признаков (Feature Importance) для Random Forest**"
    )
)

# Извлечение важности признаков из обученного Random Forest
importances = best_rf.feature_importances_
feature_names = X_train.columns

# Создание базового DataFrame
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})


# Агрегация важностей для One-Hot закодированных категориальных признаков
def aggregate_feature_names(feature_name: str) -> str:
    if (
        "_" in feature_name
        and "oldpeak" not in feature_name
        and "dataset" in feature_name
    ):
        return "Место сбора данных (dataset)"
    if "_" in feature_name and "cp" in feature_name:
        return "Тип грудной боли (cp)"

    translations = {
        "thalch": "Макс. достигнутая ЧСС (thalch)",
        "oldpeak": "Снижение сегмента ST (oldpeak)",
        "age": "Возраст (age)",
        "trestbps": "Артериальное давление в покое (trestbps)",
        "chol": "Уровень холестерина в крови (chol)",
        "exang": "Стенокардия, индуцированная физической нагрузкой (exang)",
        "sex": "Пол (sex)",
        "fbs": "Уровень сахара в крови натощак (fbs)",
        "restecg": "Результаты электрокардиографии в состоянии покоя (restecg)",
        "slope": "Наклон пика ST-сегмента во время физической нагрузки (slope)",
    }
    return translations.get(feature_name, feature_name)


importance_df["Grouped_Feature"] = importance_df["Feature"].apply(
    aggregate_feature_names
)

# Группировка и суммирование важности по агрегированным признакам
grouped_importance = (
    importance_df.groupby("Grouped_Feature")["Importance"].sum().reset_index()
)
grouped_importance = grouped_importance.sort_values(by="Importance", ascending=False)

# Визуализация
plt.figure(figsize=(14, 8))
sns.barplot(
    x="Importance",
    y="Grouped_Feature",
    data=grouped_importance,
    hue="Grouped_Feature",
    palette="mako",
    legend=False,
)

plt.title(
    "Важность признаков по мнению Random Forest (Feature Importance)",
    fontsize=14,
    pad=15,
)
plt.xlabel("Относительная важность (Сумма = 1.0)", fontsize=12)
plt.ylabel("Клинический признак", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.7)

# Добавление значений на график
for index, value in enumerate(grouped_importance["Importance"]):
    plt.text(
        value + 0.005,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="black",
    )

plt.tight_layout()
plt.show()

## **6.6. Анализ и интерпретация результатов предиктивного моделирования и Explainable AI**

**Аналитическое заключение по результатам моделирования:**
На заключительном этапе исследования был осуществлен переход от обучения «без учителя» (кластеризация) к обучению «с учителем» (Supervised Learning). Главной задачей стало создание предиктивного алгоритма для бинарной классификации пациентов (здоров/болен) на основе очищенного и валидированного признакового пространства. Оценка моделей проводилась с жестким приоритетом метрики Recall (полнота) для минимизации критических медицинских ошибок — пропуска пациентов с реальной патологией (False Negatives). Кроме того, применение методов Explainable AI (глобальной важности признаков) позволило «распаковать» логику принятия решений алгоритмом и верифицировать ее с точки зрения доказательной медицины.

**Ключевые результаты оценки:**

1.  **Триумф ансамблевых методов (Tuned Random Forest):** После оптимизации гиперпараметров с помощью `GridSearchCV` модель случайного леса (Random Forest) продемонстрировала наилучший баланс метрик: Accuracy = 0.832, F1-Score = 0.852 и ROC-AUC = 0.909. Высокий показатель площади под ROC-кривой доказывает отличную разделительную способность алгоритма при различных порогах классификации.
2.  **Феномен «стеклянного потолка» Recall:** Все исследуемые алгоритмы, включая настроенные версии, уперлись в единый предел чувствительности — Recall = 0.873 (пропуск ~13 больных пациентов из тестовой выборки). Математически это свидетельствует о том, что в наборе данных присутствует когорта пациентов со скрытым или атипичным течением болезни, чьи табличные анализы выглядят абсолютно «здоровыми». Для выявления таких случаев требуются более глубокие инструментальные исследования (например, МРТ или коронарография), выходящие за рамки базового скрининга.
3.  **Безопасный профиль ошибок (Confusion Matrix):** Анализ матрицы ошибок показал, что модель склонна ошибаться в «безопасную» сторону: количество ложных тревог (False Positives = 18) превышает количество пропусков болезни (False Negatives = 13). В контексте медицинского скрининга это оптимальное поведение алгоритма — лучше направить здорового человека на дополнительное обследование, чем проигнорировать потенциальную патологию.
4.  **Легитимизация модели через XAI (Feature Importance):** Алгоритм машинного обучения, не имея доступа к результатам наших предварительных статистических тестов, самостоятельно вывел в топ-3 самых важных предикторов наличие стенокардии напряжения (`exang`), максимальный достигнутый пульс (`thalch`) и тип грудной боли (`cp`).

**Клинико-физиологическая интерпретация паттернов:**

* **Симптомы важнее демографии:** Возраст (`age`) и пол (`sex`) оказались лишь на 6-м и 9-м местах по уровню важности для модели. Алгоритм подтвердил, что риск развития сердечно-сосудистых заболеваний определяется не паспортными данными, а объективным функциональным состоянием систем организма (переносимостью физических нагрузок, показателями ЭКГ и артериальным давлением).
* **Стресс-тесты как главный маркер:** Лидерство признаков `exang` и `thalch` подчеркивает, что патологии сердца часто маскируются в состоянии покоя и проявляются преимущественно в моменты пиковой физической нагрузки. Способность миокарда адекватно разгонять ритм без болевого синдрома — ключевой маркер, на который опирается модель при классификации пациента как «здорового».
* **Математическое подтверждение гипотез EDA:** Глобальная важность признаков `oldpeak` (депрессия ST) и `thalch` (макс. пульс) в модели Random Forest идеально совпадает с результатами многомерного дисперсионного анализа (MANOVA), где эти же факторы были признаны сильнейшими дискриминаторами между кластерами риска.

**Итоговое методологическое резюме:**

Процесс предиктивного моделирования успешно замкнул аналитический цикл проекта. Было доказано, что очищенные клинические данные позволяют построить высокоточный, устойчивый к переобучению и, главное, клинически интерпретируемый (Explainable) алгоритм. Созданная модель Random Forest не является «черным ящиком» — ее логика прозрачна, математически обоснована и полностью соответствует принципам доказательной кардиологии, что делает ее надежным прототипом для систем поддержки принятия врачебных решений (СППВР).